# Investment AI
## Notebook 06 — Build and Test Databricks AI Search

The research documents have now been parsed and transformed into semantic chunks.

This notebook creates the actual **Databricks AI Search** layer over those chunks.

```text
research.document_chunks
        ↓
Delta Sync AI Search Index
        ↓
Managed embeddings
        ↓
ANN / Hybrid Retrieval
        ↓
NEXT: Research Agent
```

The goal here is simple:

> Prove that retrieval works correctly before we give retrieval access to an agent.


## Architecture for this notebook

We will use:

- **Standard AI Search endpoint**
- **Delta Sync index**
- `chunk_id` as the primary key
- `chunk_to_embed` as the embedding source
- `chunk_to_retrieve` as the returned evidence
- Databricks managed **Qwen3 embedding model**
- metadata fields for filtering and citations
- **TRIGGERED** sync for this POC

The source Delta table already has Change Data Feed enabled from Notebook 05.


In [0]:
%pip install -q databricks-ai-search
dbutils.library.restartPython()


In [0]:
from databricks.ai_search.client import AISearchClient

CATALOG = "investment_ai"
RESEARCH_SCHEMA = "research"

SOURCE_TABLE = (
    f"{CATALOG}.{RESEARCH_SCHEMA}.document_chunks"
)

AI_SEARCH_ENDPOINT = "investment-ai-search"

AI_SEARCH_INDEX = (
    f"{CATALOG}.{RESEARCH_SCHEMA}.investment_ai_research_index"
)

EMBEDDING_MODEL = "databricks-qwen3-embedding-0-6b"

client = AISearchClient()

print("Source table :", SOURCE_TABLE)
print("Endpoint     :", AI_SEARCH_ENDPOINT)
print("Index        :", AI_SEARCH_INDEX)
print("Embedding    :", EMBEDDING_MODEL)


## 1. Create the AI Search endpoint

An AI Search endpoint provides the serving compute for one or more search indexes.

For this POC we use a **Standard** endpoint.

Databricks Free Edition currently allows one AI Search endpoint, so this project uses a single dedicated endpoint.


In [0]:
try:
    endpoint = client.get_endpoint(
        name=AI_SEARCH_ENDPOINT
    )
    print(f"Using existing endpoint: {AI_SEARCH_ENDPOINT}")

except Exception:
    endpoint = client.create_endpoint(
        name=AI_SEARCH_ENDPOINT,
        endpoint_type="STANDARD",
    )
    print(f"Creating endpoint: {AI_SEARCH_ENDPOINT}")

endpoint


## 2. Create the Delta Sync index

The index connects directly to our governed Delta chunk table.

The important mapping is:

```text
Primary key
chunk_id
     ↓

Embedding source
chunk_to_embed
     ↓
Managed embedding model
     ↓
Vector representation

Returned evidence
chunk_to_retrieve
```

We also sync the business metadata that the Research Agent will need for filtering, citations, and evidence comparison.


In [0]:
COLUMNS_TO_SYNC = [
    "chunk_to_retrieve",
    "security_id",
    "company_name",
    "sector",
    "fiscal_quarter",
    "fiscal_year",
    "document_type",
    "source_type",
    "source_authority_rank",
    "source",
    "publish_date",
    "citation_label",
    "pages_json",
    "file_name",
    "source_file_hash",
]

try:
    index = client.get_index(
        endpoint_name=AI_SEARCH_ENDPOINT,
        index_name=AI_SEARCH_INDEX,
    )
    print(f"Using existing index: {AI_SEARCH_INDEX}")

except Exception:
    index = client.create_delta_sync_index(
        endpoint_name=AI_SEARCH_ENDPOINT,
        source_table_name=SOURCE_TABLE,
        index_name=AI_SEARCH_INDEX,
        pipeline_type="TRIGGERED",
        primary_key="chunk_id",
        embedding_source_column="chunk_to_embed",
        embedding_model_endpoint_name=EMBEDDING_MODEL,
        columns_to_sync=COLUMNS_TO_SYNC,
    )
    print(f"Creating index: {AI_SEARCH_INDEX}")

index.describe()


## 3. Wait for the index to become searchable

Creating the index starts a managed synchronization pipeline.

Databricks reads the Delta rows, generates embeddings from `chunk_to_embed`, and loads the searchable index.

The first build can take several minutes.


In [0]:
import time

while True:
    description = index.describe()
    status = description.get("status", {})
    detailed_state = status.get("detailed_state", "")

    print(
        "Index state:",
        detailed_state or status.get("message", "UNKNOWN"),
    )

    if detailed_state.startswith("ONLINE"):
        break

    time.sleep(10)

print("✅ AI Search index is ONLINE.")


## 4. Semantic similarity search

Now we can search by meaning rather than exact keyword matches.

The query is converted into an embedding using the same embedding model, and AI Search finds the closest semantic chunks.


In [0]:
RETURN_COLUMNS = [
    "chunk_to_retrieve",
    "security_id",
    "company_name",
    "document_type",
    "source_type",
    "source_authority_rank",
    "publish_date",
    "citation_label",
]

semantic_results = index.similarity_search(
    query_text=(
        "Why did the compute company reduce "
        "its full-year outlook?"
    ),
    columns=RETURN_COLUMNS,
    num_results=5,
    query_type="ANN",
)

semantic_results


## 5. Hybrid search

Semantic retrieval is powerful, but investment research also contains exact terminology, company names, product names, and financial language.

**Hybrid search** combines:

- semantic vector similarity
- keyword relevance

This is often a strong default for enterprise RAG because it can capture both meaning and exact language.


In [0]:
hybrid_results = index.similarity_search(
    query_text=(
        "full-year outlook reduction "
        "supply constraints"
    ),
    columns=RETURN_COLUMNS,
    num_results=5,
    query_type="HYBRID",
)

hybrid_results


## 6. Metadata-filtered retrieval

Semantic similarity answers:

> Which chunks are relevant?

Metadata filtering adds:

> Which relevant chunks are allowed for this specific question?

Here we restrict retrieval to:

- one synthetic security
- Q2 2026
- primary research sources

This becomes useful later when the Research Agent knows the company, reporting period, or preferred source type.


In [0]:
filtered_results = index.similarity_search(
    query_text=(
        "Why was the full-year outlook reduced?"
    ),
    columns=RETURN_COLUMNS,
    filters={
        "security_id": "IA-SYN-01",
        "fiscal_quarter": "Q2",
        "fiscal_year": 2026,
        "source_type": "primary",
    },
    num_results=5,
    query_type="HYBRID",
)

filtered_results


## 7. Make the search results easier to read

The SDK returns the result rows together with a manifest describing the returned columns.

This small helper converts that response into a Spark DataFrame so we can inspect the retrieved evidence more naturally.


In [0]:
def search_results_to_df(results):
    column_names = [
        column["name"]
        for column in results["manifest"]["columns"]
    ]

    rows = results["result"]["data_array"]

    if len(rows) > 0 and len(rows[0]) == len(column_names) + 1:
        column_names = column_names + ["score"]

    return spark.createDataFrame(
        rows,
        schema=column_names,
    )


display(
    search_results_to_df(filtered_results)
)


## 8. Retrieval layer complete

We now have:

```text
Governed semantic chunks
        ↓
Delta Sync AI Search index
        ↓
Managed Qwen3 embeddings
        ↓
Semantic + hybrid retrieval
        ↓
Metadata filtering
        ↓
Cited research evidence
```

The important separation remains:

```text
AI Search
→ finds evidence

Research Agent
→ reasons over evidence
```

We deliberately built and tested retrieval **before** introducing an LLM agent.

**Next:** `07_build_research_agent.ipynb`


In [0]:
description = index.describe()

print("Endpoint :", AI_SEARCH_ENDPOINT)
print("Index    :", AI_SEARCH_INDEX)
print(
    "State    :",
    description.get("status", {}).get(
        "detailed_state",
        "UNKNOWN",
    ),
)
print("Search   : ANN + HYBRID")
print("Filters  : metadata supported")
print("\nNext: 07_build_research_agent.ipynb")
